# Support Tiers for Molecular System Forms

This document defines the support contract for molecular-system forms in the `1.x` line of MolSysMT.

Its purpose is not to list every implemented adapter. Its purpose is to make explicit:

- which forms are part of the contractual support surface;
- what kind of guarantee each tier provides;
- which forms are backed by contract verification;
- which forms are parity-verified;
- which forms are relevant to the future heavy-trajectory roadmap;
- and which areas remain outside the `1.0.0` support contract.

## How to read this document

This document separates four ideas that should not be conflated.

### 1. Support tier

The support tier tells users how strongly MolSysMT stands behind a form in the `1.x` line.

### 2. Contract verification

Contract verification means that the form is exercised by tests that validate the expected observable contract for its supported scope.

This is stronger than "the adapter exists", but different from full cross-form parity.

### 3. Parity verification

Parity verification means that equivalent molecular content represented in different supported forms is explicitly tested for equivalent results where such equivalence is part of the supported contract.

This is stronger than ordinary form support.

### 4. Heavy-mode status

Heavy-mode status indicates whether the form participates in the committed pre-`1.0.0` chunked-execution contract, is only a candidate for it, or is outside that scope.

Heavy-mode readiness must not be inferred from ordinary form support.

## Tier definitions

### Tier 1 — Contractual Forms

Tier 1 forms are part of the supported `1.x` contract.

For Tier 1 forms:

- regressions are patch-priority;
- supported semantics are expected to remain stable across the `1.x` line except for documented bug fixes and explicit support-contract revisions;
- contract support is based on implemented tests and documented scope, not only on adapter presence.

Tier 1 does not mean that every conceivable capability is guaranteed. It means that the documented supported scope of the form is part of the contractual product surface.

### Tier 2 — Supported Best-Effort Forms

Tier 2 forms are supported, maintained, and recommended where their scope is useful, but they are not part of the strongest contractual surface.

They may be:

- lossy by design;
- partially supported;
- stable in daily use without carrying full Tier 1 parity guarantees;
- likely candidates for promotion once their scope and verification harden further.

### Tier 3 — Experimental, Transitional, or Niche Forms

Tier 3 forms are available but outside the contractual core of the `1.0.0` line.

They may be:

- experimental;
- specialized;
- legacy;
- transitional;
- or insufficiently verified for contractual support.

Tier 3 forms are useful to retain, but they should not be presented as part of the guaranteed production-grade core.

In [ ]:
# Programmatic tier registry — single source of truth for runtime signals and pytest marks.
# Tier 1 forms are those present in the codebase but absent from FORM_TIERS.
# Tier 2 and 3 are listed explicitly.

from molsysmt._private.form_tier import FORM_TIERS
import molsysmt.form as _msm_form_pkg
import re
from pathlib import Path

# Discover all form directory names and their form_name
form_base = Path(_msm_form_pkg.__file__).parent
pattern = re.compile(r"^form_name\s*=\s*['\"]([^'\"]+)['\"]", re.MULTILINE)

all_forms = {}
for form_dir in sorted(form_base.iterdir()):
    if not form_dir.is_dir() or form_dir.name.startswith('_'):
        continue
    init_file = form_dir / '__init__.py'
    if not init_file.exists():
        continue
    text = init_file.read_text()
    m = pattern.search(text)
    if m:
        form_name = m.group(1)
        tier = FORM_TIERS.get(form_name, 1)
        all_forms[form_name] = tier

# Display by tier
for tier_n in [1, 2, 3]:
    forms = sorted(k for k, v in all_forms.items() if v == tier_n)
    print(f"--- Tier {tier_n} ({len(forms)} forms) ---")
    for f in forms:
        print(f"  {f}")
    print()

In [ ]:
# Sanity check: forms in FORM_TIERS but not found in form directories (stale entries)
known_dirs = set(all_forms.keys())
stale = [f for f in FORM_TIERS if f not in known_dirs]
if stale:
    print("WARNING — stale entries in FORM_TIERS (no matching form directory):")
    for f in stale:
        print(f"  {f}")
else:
    print("OK — all FORM_TIERS entries have a matching form directory.")

> **Note:** The two cells above query `molsysmt/_private/form_tier.py` programmatically and represent the live tier registry. The tables below provide additional context (Scope, Contract verification, Parity, Heavy-mode status) that cannot be auto-generated. Keep both in sync: when changing a form's tier, update `form_tier.py` and the table below in the same commit.

## Tier 1 forms

These forms are currently considered part of the contractual `1.x` support surface, within the scope stated in the notes.

| Form | Category | Scope | Contract verified | Parity verified | Heavy-mode status | Notes |
| :--- | :--- | :--- | :---: | :---: | :--- | :--- |
| `molsysmt.MolSys` | Native | Full native system object | Yes | Yes | Tier 1 target | Canonical reference form |
| `molsysmt.Topology` | Native | Native topology object | Yes | Yes | Tier 1 target | Canonical topology contract |
| `molsysmt.Structures` | Native | Native structures object | Yes | Yes | Tier 1 target | Canonical structures contract |
| `molsysmt.MolSysBuilder` | Native editable | Declared-state editable molecular system | Yes | Not applicable | Outside current heavy contract | Canonical explicit editing path |
| `molsysmt.MolSysDict` | Native declarative | Declarative in-memory molecular-system form | Yes | Partial | Outside current heavy contract | First declarative full-system form |
| `molsysmt.TopologyDict` | Native declarative | Declarative in-memory topology form | Yes | Partial | Outside current heavy contract | Declarative topology form |
| `molsysmt.StructuresDict` | Native declarative / native helper | Declarative or helper structures payload, as documented | Yes | Partial | Outside current heavy contract | Treated as part of the declarative family |
| `file:molsys_yaml` | File | Declarative YAML molecular-system file form | Yes | Partial | Outside current heavy contract | Detected by content, not by typed extension |
| `file:topology_yaml` | File | Declarative YAML topology file form | Yes | Partial | Outside current heavy contract | Detected by content, not by typed extension |
| `file:structures_yaml` | File | Declarative YAML structures file form | Yes | Partial | Outside current heavy contract | Detected by content, not by typed extension |
| `file:h5msm` | File | Native persisted molecular system | Yes | Yes | Tier 1 heavy candidate | Canonical persisted native form |
| `file:bcif` | File | BinaryCIF structural input, within documented scope | Yes | Partial | Outside current heavy contract | High-value structural form and future heavy-read candidate for structure-centric workflows |
| `file:bcif_gz` | File | Compressed BinaryCIF structural input, within documented scope | Yes | Partial | Outside current heavy contract | High-value compressed structural form and future heavy-read candidate for structure-centric workflows |
| `file:cif` | File | CIF structural input, within documented scope | Partial | Partial | Outside current heavy contract | Text CIF counterpart to `file:bcif` |
| `file:cif.gz` | File | Compressed CIF structural input, within documented scope | Partial | Partial | Outside current heavy contract | Compressed text CIF counterpart to `file:bcif_gz` |
| `file:pdb` | File | PDB file interoperability, lossy where the format is lossy | Yes | Yes within documented scope | Outside current heavy contract | Round-trip semantics explicitly constrained by PDB limitations |
| `file:xtc` | File | Supported current eager-path trajectory input | Yes | Partial | Tier 1 heavy target | Primary local trajectory file target for the first committed heavy slice |
| `file:msmpk` | File | MolSysMT packed binary format | Partial | Partial | Outside current heavy contract | Native packed format for efficient storage |
| `file:xyznpy` | File | NumPy XYZ coordinate file | Partial | Partial | Outside current heavy contract | Lightweight coordinate-only format |
| `XYZ` | Format | Simple XYZ coordinate format | Partial | Partial | Outside current heavy contract | Basic coordinate-only interoperability |
| `openmm.Topology` | Class | Topological interoperability | Yes | Yes | Outside current heavy contract | Strongly validated interop form |
| `openmm.Modeller` | Class | Editable topology-plus-positions interoperability | Partial | Partial | Outside current heavy contract | Important OpenMM preparation object |
| `openmm.Context` | Class | State extraction and interoperability within documented scope | Partial | Partial | Outside current heavy contract | Supported where explicitly documented |
| `openmm.Simulation` | Class | State-bearing interoperability within documented scope | Partial | Partial | Outside current heavy contract | Important operational object |
| `mdtraj.Trajectory` | Class | Supported current eager-path trajectory interoperability | Yes | Yes | Tier 1 heavy target | Primary trajectory interoperability target for the first committed heavy slice |
| `mdtraj.Topology` | Class | Topological interoperability | Yes | Yes | Outside current heavy contract | 384 builder tests + 15 PDB oracle tests; bugs fixed and documented in `testing_form_adapters.md` |
| `mmcif.PdbxContainers.DataContainer` | Class | mmCIF in-memory object interoperability | Partial | Partial | Outside current heavy contract | Underlying parsed object from `file:bcif` and `file:cif` workflows |
| `networkx.Graph` | Class | Topology graph interoperability | Yes | Yes | Outside current heavy contract | Contract and parity verified via MolSysBuilder: node/edge count and connectivity match source Topology exactly |
| `pdbfixer.PDBFixer` | Class | Structure preparation and repair interoperability | Partial | Partial | Outside current heavy contract | Widely used in OpenMM preparation pipelines |
| `molsysviewer.MolSysView` | Viewer | Viewer-oriented inspection and interaction | Yes | Not applicable | Outside current heavy contract | Primary MolSysSuite viewer; viewer workflow support |
| `nglview.NGLWidget` | Viewer | Viewer-oriented inspection and interaction | Yes | Yes | Outside current heavy contract | Round-trip parity verified via MolSysBuilder: MolSys → NGLWidget → MolSys preserves atom names, group names, chain count |
| `string:pdb_id` | Remote | Remote PDB retrieval entry point | Yes | Yes | Outside current heavy contract | Parity verified (network): download of 1vii matches local hp35_bcif_gz fixture on atom names, group names, chain count |
| `string:alphafold_id` | Remote | Remote AlphaFold retrieval entry point | Yes | Yes | Outside current heavy contract | Parity verified (network): download of AF-P62258-F1 round-trips losslessly through file:h5msm |
| `string:pdb_text` | String | In-memory PDB text interoperability | Yes | Yes | Outside current heavy contract | 342 getter tests + roundtrip conversion tests (Topology, MolSys, Structures, multi-model, builder fixture) |

## Tier 2 forms

These forms are supported and useful, but they remain best-effort compared with the contractual Tier 1 core.

| Form | Category | Scope | Contract verified | Parity verified | Heavy-mode status | Notes |
| :--- | :--- | :--- | :---: | :---: | :--- | :--- |
| `MDAnalysis.Universe` | Class | General MDAnalysis interoperability | Yes | Partial | Outside current heavy contract | Hardened adapter, not contractual Tier 1 yet |
| `MDAnalysis.AtomGroup` | Class | Selected-group interoperability | Yes | Partial | Outside current heavy contract | Hardened recently, still best-effort compared with Tier 1 |
| `MDAnalysis.Topology` | Class | Topology-only MDAnalysis interoperability | Partial | Partial | Outside current heavy contract | Topology-only counterpart to MDAnalysis.Universe |
| `rdkit.Mol` | Class | Chemical graph / small-molecule interoperability | Partial | Partial | Outside current heavy contract | Useful but narrower than core molecular-system forms |
| `biopython.PDBStructure` | Class | Structural biology interoperability | Partial | Partial | Outside current heavy contract | Useful and supported, but not Tier 1 contractual |
| `parmed.Structure` | Class | ParmEd interoperability | Partial | Partial | Outside current heavy contract | Stable enough for use, not Tier 1 |
| `file:mmtf` | File | MMTF binary structural file | Partial | Partial | Outside current heavy contract | Compact structural format; supported but not contractual Tier 1 |

## Tier 3 forms

These forms are available but remain outside the contractual `1.0.0` core.

| Form | Category | Scope | Contract verified | Parity verified | Heavy-mode status | Notes |
| :--- | :--- | :--- | :---: | :---: | :--- | :--- |
| `pytraj.Trajectory` | Class | Legacy / optional trajectory interoperability | Limited | No | Outside current heavy contract | Useful but not contractual |
| `pytraj.Topology` | Class | Legacy topology interoperability | Limited | No | Outside current heavy contract | Useful but not contractual |
| `biopython.Seq` | Class | Sequence-only workflows | Limited | No | Outside current heavy contract | Not a full molecular-system form |
| `biopython.SeqRecord` | Class | Annotated sequence workflows | Limited | No | Outside current heavy contract | Not a full molecular-system form |
| `file:dcd` | File | CHARMM/NAMD DCD trajectory | Limited | No | Outside current heavy contract | Trajectory-only; low-maturity adapter |
| `file:mol2` | File | MOL2 small-molecule file | Limited | No | Outside current heavy contract | Small-molecule use cases |
| `file:crd` | File | CHARMM CRD coordinate file | Limited | No | Outside current heavy contract | Coordinate-only; niche use |
| `file:inpcrd` | File | AMBER restart/coordinate file | Limited | No | Outside current heavy contract | Coordinate-only; niche use |
| `file:prmtop` | File | AMBER parameter/topology file | Limited | No | Outside current heavy contract | Topology-only; niche use |
| `file:psf` | File | CHARMM PSF topology file | Limited | No | Outside current heavy contract | Topology-only; niche use |
| `file:gro` | File | GROMACS coordinate file | Limited | No | Outside current heavy contract | Coordinate-only; niche use |
| `file:h5` | File | Generic HDF5 file | Limited | No | Outside current heavy contract | Distinct from the native `file:h5msm` form |
| `file:trjpk` | File | Trajectory pack format | Limited | No | Outside current heavy contract | Niche packed trajectory format |
| `mmtf.MMTFDecoder` | Class | MMTF in-memory decoder object | Limited | No | Outside current heavy contract | Low-level object underlying `file:mmtf` |
| `openmm.AmberInpcrdFile` | Class | OpenMM AMBER coordinate file object | Limited | No | Outside current heavy contract | Low-level OpenMM file object |
| `openmm.AmberPrmtopFile` | Class | OpenMM AMBER topology file object | Limited | No | Outside current heavy contract | Low-level OpenMM file object |
| `openmm.CharmmCrdFile` | Class | OpenMM CHARMM coordinate file object | Limited | No | Outside current heavy contract | Low-level OpenMM file object |
| `openmm.CharmmPsfFile` | Class | OpenMM CHARMM PSF file object | Limited | No | Outside current heavy contract | Low-level OpenMM file object |
| `openmm.GromacsGroFile` | Class | OpenMM GROMACS GRO file object | Limited | No | Outside current heavy contract | Low-level OpenMM file object |
| `openmm.GromacsTopFile` | Class | OpenMM GROMACS topology file object | Limited | No | Outside current heavy contract | Low-level OpenMM file object |
| `openmm.PDBFile` | Class | OpenMM PDB file object | Limited | No | Outside current heavy contract | Low-level OpenMM file object; use `file:pdb` for Tier 1 PDB support |
| `openmm.State` | Class | OpenMM simulation state | Limited | No | Outside current heavy contract | State-extraction workflows only |
| `openmm.System` | Class | OpenMM force field system | Limited | No | Outside current heavy contract | Force-field-level object; limited molecular-system semantics |
| `string:amino_acids_1` | String | One-letter amino-acid sequence string | Limited | No | Outside current heavy contract | Sequence-only input form |
| `string:amino_acids_3` | String | Three-letter amino-acid sequence string | Limited | No | Outside current heavy contract | Sequence-only input form |
| `mdtraj.DCDTrajectoryFile` | Class | mdtraj DCD file handler | Limited | No | Outside current heavy contract | Low-level; prefer `mdtraj.Trajectory` |
| `mdtraj.HDF5TrajectoryFile` | Class | mdtraj HDF5 file handler | Limited | No | Outside current heavy contract | Low-level; prefer `mdtraj.Trajectory` |
| `mdtraj.XTCTrajectoryFile` | Class | mdtraj XTC file handler | Limited | No | Outside current heavy contract | Low-level; prefer `mdtraj.Trajectory` or `file:xtc` |
| `molsysmt.CIFFileHandler` | Class | Internal CIF file handler | Limited | No | Outside current heavy contract | Internal handler; prefer `file:cif` |
| `molsysmt.GROFileHandler` | Class | Internal GRO file handler | Limited | No | Outside current heavy contract | Internal handler; prefer `file:gro` |
| `molsysmt.MolecularMechanics` | Class | Molecular mechanics parameters object | Limited | No | Outside current heavy contract | Outside `1.0.0` contract scope |
| `molsysmt.MolecularMechanicsDict` | Class | Declarative molecular mechanics parameters | Limited | No | Outside current heavy contract | Outside `1.0.0` contract scope |
| `molsysmt.ViewerJSON` | Class | JSON-based viewer payload | Limited | No | Outside current heavy contract | Internal viewer serialization object |

## Explicitly outside the `1.0.0` support contract

The following area is explicitly outside the `1.0.0` support contract, even if code remains in the repository:

- `molsysmt.molecular_dynamics/**`

This means:

- it is not part of the contractual `1.0.0` product surface;
- it does not define the release baseline;
- and it is excluded from the stabilization-oriented coverage target.

## Contract verification status

Contract verification for this document should be interpreted conservatively.

The relevant question is not whether a form can be imported or whether some conversion exists. The relevant question is whether the supported scope of the form is exercised by tests that validate the expected user-visible behavior.

The contract-verification program should keep hardening:

- native forms;
- declarative forms;
- canonical persisted forms;
- and the strongest interoperability forms that are part of Tier 1.

## Parity verification

Parity verification is narrower than general support.

The expected parity matrix should be derived from the support contract, not guessed informally. In practice, this means:

- Tier 1 forms should carry the strongest parity obligations where equivalent semantics are meaningful;
- Tier 2 forms may be parity-verified only for their documented scope;
- viewer and remote forms may remain lossy or best-effort where that is intrinsic to their role.

Two parity axes should remain explicit:

1. **form parity**
   - equivalent molecular content represented in different forms should produce equivalent observable results where the contract says so;
2. **execution parity**
   - eager and heavy execution paths should produce equivalent results for operations that officially support both.

## Heavy-mode status

Heavy-mode support should not be inferred from ordinary form support.

This document therefore uses the following heavy-mode states informally:

- `Tier 1 heavy target`
  - forms that are expected to matter directly for the committed pre-`1.0.0` heavy slice;
- `heavy candidate`
  - forms that are plausible heavy inputs but are not yet committed in the support contract;
- `outside current heavy contract`
  - forms that may still be fully supported for ordinary workflows while not participating in the first heavy slice.

The authoritative design for heavy trajectories is tracked in:

- `devguide/scalability_and_heavy_trajectories_v2.md`

The support contract and the heavy roadmap must remain aligned, but they are not the same document.

## Contractual capability matrix

This matrix summarizes the expected capability envelope of each support tier. It is intentionally high-level: the authoritative form-by-form scope remains the tier tables above, but this section provides the product-level view that users need when deciding whether a form is suitable for production work.

| Capability | Tier 1 (Contractual forms) | Tier 2 (Supported best-effort forms) | Tier 3 (Experimental / niche forms) |
| :--- | :--- | :--- | :--- |
| **Basic introspection** (`get`, `info`, documented `compare`) | Full, within documented form scope | Full or near-full within documented scope | Limited and form-dependent |
| **Selection semantics** (`select`) | Full where selection is part of the form contract | Partial to full, depending on form scope | Best-effort |
| **Structural analysis** (`distances`, `RMSD`, related structure operations) | Full on the native/core surface; heavy support only where explicitly declared | Eager-path support where documented; heavy not implied | Best-effort |
| **Topology editing** | Full through `MolSysBuilder` / `build.editable(...)` on the native editing path | Partial where the ecosystem form can be converted and edited safely | Not contractual |
| **Coordinate updates** (`set`) | Full within documented structural scope | Partial to full within documented scope | Limited |
| **Format conversion** (`convert`) | Expected to preserve the documented supported scope; lossless where the form pair is contractually lossless | Supported but may be lossy or partial by design | Experimental or transitional |
| **Visualization workflows** (`view`, viewer-oriented adapters) | Verified where viewing is part of the documented workflow | Supported where explicitly documented | Limited |
| **Heavy / chunked execution** | Only for forms explicitly marked as Tier 1 heavy targets in this document | Not contractual unless promoted explicitly | Outside current heavy contract |

Notes:

- "Full" never means "all imaginable semantics". It means full support for the documented contractual scope of that tier.
- Lossy formats remain Tier 1 when the lossy boundary is intrinsic to the format and explicitly documented, as in the case of PDB-based workflows.
- Heavy-mode support must be read from the form-specific heavy-status column, not inferred from the general support tier alone.
- The goal of this matrix is expectation management and testing focus. The detailed contractual source of truth remains the per-form tables above.

## Runtime tier signals

Since March 2026, MolSysMT communicates support-tier classification at runtime using
SMonitor structured diagnostics.  This is implemented via the **support-tier protocol**
documented in `devguide/support_tier_protocol.md`.

Key behavior:

- **Tier 1** forms and functions produce **no runtime signal** (silence by design).
- **Tier 2** forms and functions emit a **WARNING** the first time they are used per
  session.
- **Tier 3** forms and functions emit an **INFO** signal the first time they are used
  per session.

Signals are deduplicated: each form/function fires at most once per Python session,
regardless of how many times it is used.  This prevents log pollution in loops.

The signal for forms is injected in `molsysmt/basic/get_form.py`, which is called by
every public API function.  The per-form tier mapping lives in
`molsysmt/_private/form_tier.py`.

Functions are decorated individually with `@support_tier(N)` from
`molsysmt._private.smonitor`.

---

## Tier 1 guarantee summary

Tier 1 forms are the forms that MolSysMT is prepared to defend as part of its supported `1.x` line.

This means, in practical terms:

- regressions are patch-priority;
- Tier 1 regressions block the release of new minor versions until resolved or explicitly reclassified;
- regressions discovered in the wild should trigger an immediate patch-release decision path;
- semantics are expected to remain stable across `1.x` except for documented bug corrections and explicit contract revisions;
- support claims must be backed by tests, not only by adapters;
- any future heavy-mode commitments for Tier 1 forms must be reflected explicitly in this document rather than inferred informally.